# ORION — 가속기 실측 핸드오프 (Kaggle · Colab · Local)

**셀을 위에서부터 순서대로 실행**하면, 이 세션의 가속기에서 ORION 두 비율
(**R_C** 잔류, **R_B** 오버랩)을 **실측**하고 결과를 JSON + zip으로 저장합니다.
Kaggle · Colab · 로컬 Jupyter를 **자동 감지**하며, 측정 엔진이 노트북에 **내장**되어
있어 인터넷/클론 없이도 동작합니다.

> ⚠️ **Kaggle 사용자:** 오른쪽 **Settings → Accelerator → GPU T4 x2** 를 선택하세요.
> (Internet 은 켜지 않아도 됩니다 — 엔진 내장.) GPU면 `torch.cuda.Event` 로 실측하고,
> GPU가 없으면 자동으로 CPU(NumPy)로 축소 실행됩니다.


## STEP 0 — 환경 감지 + (선택) 저장소 클론

In [ ]:
import os, sys, subprocess

# 출력 베이스 디렉터리 자동 감지 (Kaggle / Colab / 로컬)
if os.path.isdir('/kaggle/working'):
    BASE = '/kaggle/working'
elif os.path.isdir('/content'):
    BASE = '/content'
else:
    BASE = os.getcwd()
print('BASE =', BASE)

# 저장소는 provenance 용으로만 시도(실패해도 계속). 측정 엔진은 아래에 내장됨.
REPO = os.path.join(BASE, 'orion')
if not os.path.isdir(REPO):
    try:
        subprocess.run(['git', 'clone', '--depth', '1',
                        'https://github.com/leemgs/orion', REPO],
                       timeout=120, check=False)
    except Exception as e:
        print('clone 건너뜀(무관):', e)
print('repo present:', os.path.isdir(REPO))

# numpy 는 폴백 경로용(Kaggle 에는 기본 설치). torch 는 GPU 이미지에 기본 설치.
try:
    import numpy  # noqa
except Exception:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'numpy'], check=False)
print('setup done')


## STEP 1 — 가속기 확인 (T4 x2 기대)

In [ ]:
try:
    import torch
    print('torch', torch.__version__, '| cuda available:', torch.cuda.is_available())
    if torch.cuda.is_available():
        for i in range(torch.cuda.device_count()):
            print(f'  GPU{i}:', torch.cuda.get_device_name(i))
except Exception as e:
    print('torch 없음 -> CPU(NumPy) 폴백으로 실행됩니다:', e)
try:
    print(subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total',
                          '--format=csv,noheader'], capture_output=True,
                         text=True, timeout=30).stdout.strip() or '(nvidia-smi 출력 없음)')
except Exception:
    pass


## STEP 2 — 측정 엔진 (내장, 자동 감지)

In [ ]:
import json, math, platform, statistics, time
from pathlib import Path
NAN = float("nan"); SEED = 20260818

class NumpyBackend:
    name, device = "numpy", "CPU(numpy)"
    def __init__(self):
        import numpy as np
        self.np = np; self.rng = np.random.default_rng(SEED)
    def new_weight(self, d):
        return (self.rng.standard_normal((d, d), dtype="float32") / self.np.float32(d ** 0.5))
    new_host_weight = new_weight
    def transfer_in(self, hw):
        t0 = time.perf_counter(); dev = hw.copy(); return dev, time.perf_counter() - t0
    def matmul(self, x, w, r):
        t0 = time.perf_counter(); y = x
        for _ in range(r): y = self.np.matmul(y, w)
        if y.shape[0] < 0: raise RuntimeError
        return y, time.perf_counter() - t0
    def new_activation(self, b, d): return self.rng.standard_normal((b, d), dtype="float32")

class TorchBackend:
    name = "torch"
    def __init__(self, want_tpu=False):
        import torch; self.torch = torch; self.is_xla = False
        if want_tpu:
            import torch_xla.core.xla_model as xm
            self.xm = xm; self.dev = xm.xla_device(); self.is_xla = True; self.device = "TPU(XLA)"
        elif torch.cuda.is_available():
            self.dev = torch.device("cuda"); self.device = f"GPU:{torch.cuda.get_device_name(0)}"
        else:
            self.dev = torch.device("cpu"); self.device = "CPU(torch)"
        torch.manual_seed(SEED)
    def _sync(self):
        if self.is_xla: self.xm.mark_step(); self.xm.wait_device_ops()
        elif self.dev.type == "cuda": self.torch.cuda.synchronize()
    def new_weight(self, d):
        return self.torch.randn(d, d, device=self.dev, dtype=self.torch.float32) / (d ** 0.5)
    def new_host_weight(self, d):
        w = self.torch.randn(d, d, dtype=self.torch.float32) / (d ** 0.5)
        if self.dev.type == "cuda": w = w.pin_memory()
        return w
    def transfer_in(self, hw):
        t = self.torch
        if self.dev.type == "cuda":
            s = t.cuda.Event(enable_timing=True); e = t.cuda.Event(enable_timing=True)
            t.cuda.synchronize(); s.record()
            dev = hw.to(self.dev, non_blocking=True); e.record(); t.cuda.synchronize()
            return dev, s.elapsed_time(e) / 1e3
        t0 = time.perf_counter(); dev = hw.to(self.dev); self._sync()
        return dev, time.perf_counter() - t0
    def matmul(self, x, w, r):
        t = self.torch
        if self.dev.type == "cuda":
            s = t.cuda.Event(enable_timing=True); e = t.cuda.Event(enable_timing=True)
            t.cuda.synchronize(); s.record(); y = x
            for _ in range(r): y = y @ w
            e.record(); t.cuda.synchronize(); return y, s.elapsed_time(e) / 1e3
        t0 = time.perf_counter(); y = x
        for _ in range(r): y = y @ w
        self._sync(); return y, time.perf_counter() - t0
    def new_activation(self, b, d):
        return self.torch.randn(b, d, device=self.dev, dtype=self.torch.float32)

def make_backend():
    try:
        import torch_xla.core.xla_model  # noqa
        return TorchBackend(want_tpu=True)
    except Exception: pass
    try:
        import torch  # noqa
        return TorchBackend(want_tpu=False)
    except Exception:
        return NumpyBackend()

def measure_point(be, d, n_layers, batch, resident_frac, comp_repeats, n_windows):
    n_res = max(0, min(n_layers, round(resident_frac * n_layers))); n_off = n_layers - n_res
    resident = [be.new_weight(d) for _ in range(n_res)]
    host_off = [be.new_host_weight(d) for _ in range(n_off)]
    def one_step():
        x = be.new_activation(batch, d); t0 = time.perf_counter(); tx = comp = 0.0
        for hw in host_off:
            dev_w, dt = be.transfer_in(hw); tx += dt
            x, ct = be.matmul(x, dev_w, comp_repeats); comp += ct
        for rw in resident:
            x, ct = be.matmul(x, rw, comp_repeats); comp += ct
        return time.perf_counter() - t0, comp, tx
    one_step()  # warm-up (excluded)
    tots, comps, txs = [], [], []
    for _ in range(n_windows):
        t, c, x = one_step(); tots.append(t); comps.append(c); txs.append(x)
    t_comp = statistics.mean(comps); t_tx = statistics.mean(txs)
    r_c = n_res / n_layers if n_layers else NAN
    r_b = (t_comp / t_tx) if t_tx > 0 else NAN
    return {"r_c": r_c, "t_comp_s": t_comp, "t_transfer_s": t_tx,
            "t_total_s": statistics.mean(tots), "r_b": r_b,
            "t_total_sd": statistics.pstdev(tots)}

def classify(r_c, r_b, tc=0.50, tb=1.0):
    if r_c < tc: return "capacity-limited"
    if r_b == r_b and r_b < tb: return "io-limited"
    return "coordination-dominated"

def json_safe(v):
    if isinstance(v, float) and not math.isfinite(v): return None
    if isinstance(v, dict): return {k: json_safe(x) for k, x in v.items()}
    if isinstance(v, list): return [json_safe(x) for x in v]
    return v

print("engine loaded")


## STEP 3 — 실측 실행 (R_C / R_B 스윕)  ⭐

In [ ]:
be = make_backend()
is_accel = getattr(be, "device", "").startswith("GPU") or getattr(be, "is_xla", False)
if is_accel:
    D, LAYERS, BATCH, WINDOWS, COMP = 2048, 24, 8, 10, 6
    RC_GRID = [0.1, 0.2, 0.3, 0.4, 0.45, 0.5, 0.55, 0.6, 0.75, 0.9, 1.0]
    RB_REPS = [1, 2, 3, 4, 6, 8, 12]
else:
    D, LAYERS, BATCH, WINDOWS, COMP = 512, 12, 8, 5, 3
    RC_GRID = [0.1, 0.25, 0.4, 0.5, 0.6, 0.75, 1.0]
    RB_REPS = [1, 2, 4, 8]

print(f"[measure] backend={be.name} device={be.device} d={D} layers={LAYERS} "
      f"batch={BATCH} windows={WINDOWS}")
summary = []
print("\n=== Sweep A: R_C (residency) ===")
print(f"{'R_C':>6} {'regime':>22} {'T_total(ms)':>13} {'R_B':>8}")
for rc in RC_GRID:
    m = measure_point(be, D, LAYERS, BATCH, rc, COMP, WINDOWS)
    reg = classify(m["r_c"], m["r_b"])
    print(f"{m['r_c']:>6.2f} {reg:>22} {m['t_total_s']*1e3:>13.2f} {m['r_b']:>8.2f}")
    m.update(sweep="R_C", regime=reg, device=be.device); summary.append(m)
print("\n=== Sweep B: R_B (overlap) ===")
print(f"{'reps':>6} {'regime':>22} {'T_total(ms)':>13} {'R_B':>8}")
for rep in RB_REPS:
    m = measure_point(be, D, LAYERS, BATCH, 0.6, rep, WINDOWS)
    reg = classify(m["r_c"], m["r_b"])
    print(f"{rep:>6} {reg:>22} {m['t_total_s']*1e3:>13.2f} {m['r_b']:>8.2f}")
    m.update(sweep="R_B", regime=reg, comp_repeats=rep, device=be.device); summary.append(m)
print("\nsweeps done")


## STEP 4 — 결과 저장 + paste-back + zip 다운로드

In [ ]:
slug = "".join(c.lower() if c.isalnum() else "-" for c in be.device)
slug = "-".join(filter(None, slug.split("-")))
payload = {"device": be.device, "backend": be.name, "d": D, "n_layers": LAYERS,
           "batch": BATCH, "windows": WINDOWS, "python": platform.python_version(),
           "theta_C": 0.50, "theta_B": 1.0, "points": summary}
out = Path(BASE)
fp = out / f"orion_accel_{slug}.json"
fp.write_text(json.dumps(json_safe(payload), indent=2, allow_nan=False))

a = [s for s in summary if s["sweep"] == "R_C"]
cap = [s["t_total_s"] for s in a if s["r_c"] < 0.5]
res = [s["t_total_s"] for s in a if s["r_c"] >= 1.0]
print("=" * 62)
print("[paper paste-back — 이 블록을 그대로 전달해 주세요]")
print(f"device = {be.device}")
if cap and res:
    print(f"capacity-limited(R_C<0.5) mean T_total = {statistics.mean(cap)*1e3:.2f} ms")
    print(f"resident(R_C>=1.0)        mean T_total = {statistics.mean(res)*1e3:.2f} ms")
    print(f"capacity crossover        = {statistics.mean(cap)/statistics.mean(res):.2f}x")
print("=" * 62)

# 그래프(있으면) + zip
imgs = []
try:
    import matplotlib.pyplot as plt
    aa = sorted(a, key=lambda s: s["r_c"])
    bb = sorted([s for s in summary if s["sweep"] == "R_B" and s["r_b"] == s["r_b"]],
                key=lambda s: s["r_b"])
    fig, ax = plt.subplots(1, 2, figsize=(11, 4))
    ax[0].plot([s["r_c"] for s in aa], [s["t_total_s"]*1e3 for s in aa], "o-")
    ax[0].axvline(0.5, ls="--", c="r"); ax[0].set_xlabel("R_C"); ax[0].set_ylabel("T_total (ms)")
    ax[0].set_title("Sweep A: residency")
    ax[1].plot([s["r_b"] for s in bb], [s["t_total_s"]*1e3 for s in bb], "s-")
    ax[1].axvline(1.0, ls="--", c="r"); ax[1].set_xlabel("R_B"); ax[1].set_ylabel("T_total (ms)")
    ax[1].set_title("Sweep B: overlap")
    plt.tight_layout(); png = out / "orion_sweeps.png"; plt.savefig(png, dpi=120); plt.show()
    imgs.append(png)
except Exception as ex:
    print("plot skipped:", ex)

import zipfile
zp = out / "orion_results.zip"
with zipfile.ZipFile(zp, "w", zipfile.ZIP_DEFLATED) as z:
    z.write(fp, fp.name)
    for im in imgs: z.write(im, im.name)
print("saved:", fp)
print("zip  :", zp, "  (Kaggle: Output 패널에서 다운로드)")
